In [ ]:
import numpy as np
import pandas as pd


from sklearn.preprocessing import StandardScaler

from collections import Counter

import math
import csv

mcmcseed=1
rng = np.random.default_rng(mcmcseed)

from scipy.stats import multivariate_normal, invwishart, wishart, dirichlet, rv_discrete, beta, gamma

import sys
print(sys.version)


q=1
p=1
d_1=3 
Ktrue=3 


import argparse


dataseedtrain=35
N=5000

param_rng = np.random.default_rng(seed=123456)  


truemu_kg=np.zeros([Ktrue,1,q])


truemu_kg[:,0,0]=np.array([0,2,5])


trueSigma_k=np.zeros([Ktrue,1,q,q])


trueSigma_k_diag=np.array([0.1,0.2,0.5])

for k in range(0,Ktrue):
    
        
        trueSigma_k[k,0,:]=trueSigma_k_diag[k] 
        
        
trueclusterprob = np.array([0.4,0.35,0.25]) 
trueclusterprob /= trueclusterprob.sum()


categorysizes=[d_1]
truepsi=np.zeros([Ktrue,p]+[d_1]) 


highprob=0.6
truepsi[:,0,:]=(1-highprob)/(d_1-1)+np.zeros((Ktrue,Ktrue))
for k in range(0,Ktrue):
    truepsi[k,0,k] = highprob
        
def generate_data_contcase(N,Ktrue,truepsi,truemu_kg,trueSigma_k,trueclusterprob, data_seed):
    
    data_rng = np.random.default_rng(seed=data_seed)
    
    component = data_rng.choice(Ktrue, size=N, p=trueclusterprob)
    
    categoricaldata = np.zeros(N, dtype=int)

    for i in range(N):
        
            
            categoricaldata[i] = data_rng.choice(np.arange(categorysizes[0]), p=truepsi[component[i],0,:] )

                                                                      
    continuousdata=np.zeros([N,q])

    for i in range(0,N):


        x =  data_rng.multivariate_normal(truemu_kg[component[i],0,:], trueSigma_k[component[i],0,:], size=1)

        continuousdata[i,:]=x
        
    return continuousdata, categoricaldata, component


continuousdata, categoricaldata, component=generate_data_contcase(N,Ktrue,truepsi,truemu_kg,trueSigma_k,trueclusterprob, data_seed=dataseedtrain)


fulldatamat=np.hstack((continuousdata,np.reshape(categoricaldata,(N,1)), np.reshape(component,(N,1)) )) 

contcol_list = [f'contcol_{i}' for i in range(1, q+1)]
catcol_list=[f'catcol_{i}' for i in range(1,p+1)]

fulldata=pd.DataFrame(fulldatamat, columns = contcol_list+catcol_list+['clusterlabel'])    


discrete_cols=catcol_list+['clusterlabel'] 

for col in discrete_cols:
    fulldata[col] = fulldata[col].astype(int)


allcategoricalvalues=[0,1]


modelled_continuous_columns=contcol_list 
modelled_categorical_columns=catcol_list


def merge_columns(row):
    return '_'.join(row.astype(str))


fulldata['combinedcategories'] = fulldata[modelled_categorical_columns].apply(merge_columns, axis=1)


modelled_categorical_columns_combined=['combinedcategories']


fulldataframe=pd.DataFrame(fulldata, columns = modelled_continuous_columns+modelled_categorical_columns+modelled_categorical_columns_combined+['clusterlabel'])


scaler = StandardScaler()

fulldataframe[modelled_continuous_columns] = scaler.fit_transform(fulldataframe[modelled_continuous_columns])

scaler.fit_transform(np.array(fulldata[modelled_continuous_columns])) 

combinedcategories=np.array(fulldataframe['combinedcategories'])

continuousdata=np.array(fulldataframe[modelled_continuous_columns])

allcatvals=np.unique(combinedcategories,return_counts=True)


fulldataframegroupbymeans = fulldataframe.groupby(['combinedcategories'], as_index=False)[modelled_continuous_columns].mean() 

fulldataframegroupbystd = fulldataframe.groupby(['combinedcategories'], as_index=False)[modelled_continuous_columns].std() 


categorical_columns_dims=[]

for catcols in modelled_categorical_columns:
    categorical_columns_dims.append(len(fulldataframe[catcols].unique()))


print('done simulating data')

print('check true parameter values')

print('truemu_kg')
print(truemu_kg)

print('trueSigma_kg')
print(trueSigma_k)

print('truepsi')
print(truepsi)

print('trueclusterprob')
print(trueclusterprob)

Ntest=min(int(N*0.4),2000)

testcontinuousdata, testcategoricaldata, testcomponent=generate_data_contcase(Ntest,Ktrue,truepsi,truemu_kg,trueSigma_k,trueclusterprob, data_seed=dataseedtrain+1000)


testdatamat=np.hstack((testcontinuousdata,np.reshape(testcategoricaldata,(Ntest,1)), np.reshape(testcomponent,(Ntest,1)) ))

contcol_list = [f'contcol_{i}' for i in range(1, q+1)]

testdata=pd.DataFrame(testdatamat, columns = contcol_list+catcol_list+['clusterlabel'])   

for col in catcol_list+['clusterlabel']:
    testdata[col] = testdata[col].astype(int)

testdata['combinedcategories'] = testdata[modelled_categorical_columns].apply(merge_columns, axis=1)


testdataframe=pd.DataFrame(testdata, columns = modelled_continuous_columns+modelled_categorical_columns+modelled_categorical_columns_combined+['clusterlabel'])

scalertest = StandardScaler()

testdataframe[modelled_continuous_columns] = scalertest.fit_transform(testdataframe[modelled_continuous_columns])

testcontinuousdata=np.array(testdataframe[modelled_continuous_columns])

print('done')

import pickle

outputtruemu_kg = open('truemu_kg'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(truemu_kg, outputtruemu_kg)
outputtruemu_kg.close()

outputtrueSigma_k = open('trueSigma_k'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(trueSigma_k, outputtrueSigma_k)
outputtrueSigma_k.close()

outputtruepsi = open('truepsi'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(truepsi, outputtruepsi)
outputtruepsi.close()

outputtrueclusterprob = open('trueclusterprob'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(trueclusterprob, outputtrueclusterprob)
outputtrueclusterprob.close()

outputfulldata=open('trainingdata'+str(dataseedtrain)+'.pkl','wb')
pickle.dump(fulldata,outputfulldata)
outputfulldata.close()

outputtestdata=open('testdata'+str(dataseedtrain)+'.pkl','wb')
pickle.dump(testdata,outputtestdata)
outputtestdata.close()

outputscaler=open('scaler'+str(dataseedtrain)+'.pkl','wb')
pickle.dump(scaler,outputscaler)
outputscaler.close()

print('done saving true params')

In [ ]:
Psiindependentofk=0

catvaluetoindex = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))

mudependentong=0
Sigmadependentong=0
pidependentong=0

if mudependentong==0:
    catvaluetoindexmu = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) )) 
    
    allcatvalsmu=np.array(['0'])
    
    
    catvaluetoindexmu['0']=0
    combinedcategoriesmu=np.array(['0']*N)
else:
    allcatvalsmu=allcatvals
    catvaluetoindexmu = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriesmu=combinedcategories
    
if Sigmadependentong==0:
    catvaluetoindexSigma = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) ))
    allcatvalsSigma=np.array(['0'])
    
    catvaluetoindexSigma['0']=0
    combinedcategoriesSigma=np.array(['0']*N)
else:
    allcatvalsSigma=allcatvals
    catvaluetoindexSigma = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriesSigma=combinedcategories
    
if pidependentong==0:
    catvaluetoindexpi = dict(zip(allcatvals[0], [0] * len(allcatvals[0]) ))
    allcatvalspi=np.array(['0'])
    
    catvaluetoindexpi['0']=0
    combinedcategoriespi=np.array(['0']*N)
else:
    allcatvalspi=allcatvals
    catvaluetoindexpi = dict(zip(allcatvals[0], list(range(0, len(allcatvals[0])))))
    combinedcategoriespi=combinedcategories

print('check mu catval variables')
print(allcatvalsmu)
print(catvaluetoindexmu)
print(Counter(combinedcategoriesmu))

print('check Sigma catval variables')
print(allcatvalsSigma)
print(catvaluetoindexSigma)
print(Counter(combinedcategoriesSigma))

print('check pi catval variables')
print(allcatvalspi)
print(catvaluetoindexpi)
print(Counter(combinedcategoriespi))

In [ ]:
from scipy.stats import multivariate_normal, invwishart, wishart, dirichlet, rv_discrete, beta, gamma


K = Ktrue
p = len(modelled_categorical_columns)
q = len(modelled_continuous_columns) 


checkrow = 3
print(combinedcategories[checkrow])
print(catvaluetoindex[combinedcategories[checkrow]])

iterations = 3001 
burn_in = int(iterations * 0.2)

mdims = tuple([iterations,K] + [len(allcatvalsmu[0])] + [q]) 
betadims=tuple([iterations,K] + [len(allcatvalsmu[0])]) 

beta_kg_prior=1.0


z = np.zeros((N, 1))

zprobnormalised = np.zeros((N, K))+1/K

from kmodes.kprototypes import KPrototypes

cat_indices = [fulldataframe[modelled_continuous_columns+modelled_categorical_columns].columns.get_loc(col) for col in modelled_categorical_columns] 

kproto = KPrototypes(n_clusters=K, init='Cao', n_jobs=-1, random_state=42)
kprotoclusters = kproto.fit_predict(fulldataframe[modelled_continuous_columns+modelled_categorical_columns].to_numpy(), categorical=cat_indices)

z[:,0]=kprotoclusters

z_history = np.zeros((iterations, N), dtype=np.int8 if K < 128 else np.int16)
z_history[0] = z[:, 0]        

mu = np.zeros(mdims)  


Sigma = np.zeros((iterations, K, len(allcatvalsSigma[0]), q, q))


piprobdim = tuple([iterations]+ [K]) 


piprob = np.zeros(piprobdim)  

Psidims = tuple([iterations, K, p] + [np.max(categorical_columns_dims)]) 


Psi = np.zeros(Psidims)  


Phi = 0.25*np.eye(q) 
nu = q + K + 1


mu_cs0_all = np.zeros((tuple([len(allcatvalsmu[0])] + [continuousdata.shape[1]]))) 

print(np.shape(mu_cs0_all))

if mudependentong==0:
    for j in allcatvalsmu[0]: 
        
        mu_cs0 = np.zeros(q) 
        

        mu_cs0_all[catvaluetoindexmu[j], :] = mu_cs0 
    
else:
    for j in allcatvalsmu[0]: 
        
        mu_cs0 = np.zeros(q) 
        mu_cs0 = np.array(mu_cs0)[0, :] 

        mu_cs0_all[catvaluetoindexmu[j], :] = mu_cs0 

print('check mu_cs0_all')
print(mu_cs0_all)


piparamprior=1/K 


for g in allcatvalsmu[0]: 
    piprob[0, :] = rng.dirichlet(alpha=np.ones(K)*piparamprior, size=1)  
    
if mudependentong==0:
    for k in range(0,K):
        for g in allcatvalsmu[0]: 
            Sigma[0, k, catvaluetoindexSigma[j], :] = invwishart.rvs(df=nu, scale=Phi, size=1, random_state=rng)  
            print('j is ' + str(j) + ' k is ' + str(k)) 
            
            mu_cs0 = mu_cs0_all[catvaluetoindexmu[j], :] 
            
           
            print('check mu_cs0') 
            print(mu_cs0) 
            print(np.shape(mu_cs0)) 
            print(np.abs(mu_cs0)) 
            print('check Sigma_cs0') 
            print(np.diag(np.abs(mu_cs0))) 
            mu[0, k, catvaluetoindexmu[j], :] = rng.multivariate_normal(mean=mu_cs0,cov=Sigma[0, k, catvaluetoindexSigma[j], :]/(beta_kg_prior),size=1)  

else:
    for k in range(K):
        
        for j in allcatvalsmu[0]: 
            Sigma[0, k, catvaluetoindexSigma[j], :] = invwishart.rvs(df=nu, scale=Phi, size=1, random_state=rng)  
            print('j is ' + str(j) + ' k is ' + str(k)) 
            
            mu_cs0 = mu_cs0_all[catvaluetoindexmu[j], :] 
            mu_cs0 = np.array(mu_cs0)[0, :] 

            print('check mu_cs0') 
            print(mu_cs0) 
            print(np.shape(mu_cs0)) 
            print(np.abs(mu_cs0)) 
            print('check Sigma_cs0') 
            print(np.diag(np.abs(mu_cs0))) 
            mu[0, k, catvaluetoindexmu[j], :] = rng.multivariate_normal(mean=mu_cs0,cov=Sigma[0, k, catvaluetoindexSigma[j], :]/(beta_kg_prior),size=1)  


for k in range(0,K): 
    for j in range(0,p): 
        
        Psi[0, k, j, 0:categorical_columns_dims[j]] = rng.dirichlet(alpha=np.ones(categorical_columns_dims[j]) / categorical_columns_dims[j], size=1) 
            
                
print('check piprob')
print(piprob[0, :])
print('check Psi')
print(Psi[0, :])
print('check mu')
print(mu[0, :])
print('check Sigma')
print(Sigma[0, :])

In [ ]:
#read saved outputs instead of running Gibbs sampler.

with open('mu'+str(dataseedtrain)+'.pkl', 'rb') as file:
    mu=pickle.load(file)


if mu.ndim == 5: 
    mu = np.squeeze(mu, axis=3) 

with open('zprob'+str(dataseedtrain)+'.pkl', 'rb') as file:
    zprobnormalised=pickle.load(file)

with open('z'+str(dataseedtrain)+'.pkl', 'rb') as file:
    z=pickle.load(file)

with open('Psi'+str(dataseedtrain)+'.pkl', 'rb') as file:
    Psi=pickle.load(file)


if Psi.ndim == 5: 
    Psi = np.squeeze(Psi, axis=3) 

with open('Sigma'+str(dataseedtrain)+'.pkl', 'rb') as file:
    Sigma=pickle.load(file)

with open('piprob'+str(dataseedtrain)+'.pkl', 'rb') as file:
    piprob=pickle.load(file)


if piprob.ndim == 3: 
    piprob = np.squeeze(piprob, axis=1) 

with open('iteration'+str(dataseedtrain)+'.pkl', 'rb') as file:
    iteration=pickle.load(file)


In [ ]:
import pickle
import time
start_time = time.time()

def computejoint(z,mu,Sigma,Psi,pi):

    zint = z.astype(int)

    gvec = np.empty(N)
    for k in range(0,K):
        idx = np.where(zint == k)[0]
        if idx.size:
            gvec[idx] = multivariate_normal.logpdf(x=continuousdata[idx], mean=mu[k,:], cov=Sigma[k,:])


    xllh = gvec.sum()
    muprior=0
    Sigmaprior=0
    for k in range(0,K):
        muprior=muprior+multivariate_normal.logpdf(x=mu[k,:],mean=mu_cs0_all[0,:],cov=Sigma[k,:]/beta_kg_prior)
        Sigmaprior=Sigmaprior+invwishart.logpdf(x=Sigma[k,:], df=nu, scale=Phi)


    cllh=0
    for j in range(0,p):

        cllh=cllh+np.log(Psi[zint, j, catcodes[:, j]]).sum()


    psiprior=0
    for k in range(0,K):
        for j in range(0,p):
            psiprior=psiprior+dirichlet.logpdf(x=Psi[k,j,:],alpha=np.ones(categorical_columns_dims[j]) / categorical_columns_dims[j])


    zllh=np.log(pi[zint]).sum()
    piprior=dirichlet.logpdf(x=pi, alpha=np.ones(K)*piparamprior)
    return xllh+muprior+Sigmaprior+cllh+psiprior+zllh+piprior


timerecord=np.zeros(iterations)
jointdistval=np.zeros(iterations)

catcodes = fulldataframe[modelled_categorical_columns].to_numpy().astype(int)  

gibbs_compute_elapsed = 0.0

for iteration in range(1, iterations):
    print('we are at iteration')
    print(iteration)

    iter_t0 = time.perf_counter()   

    for clusterno in range(0,K):
        for j in range(0,p):
                alphatemp = np.zeros(categorical_columns_dims[j])
                for g_j in range(0,categorical_columns_dims[j]):

                    count_kgj = (z[:, 0] == clusterno) & (fulldataframe[modelled_categorical_columns[j]] == g_j)
                    alphatemp[g_j] = 1 / categorical_columns_dims[j] + np.sum(count_kgj)


                Psi[iteration, clusterno, j, 0:categorical_columns_dims[j]] = rng.dirichlet(alpha=alphatemp, size=1)


    for clusterno in range(K):

        Phi_new = Phi

        for catvalue in allcatvalsSigma[0]:
            count_kc = (z[:, 0] == clusterno) & (combinedcategoriesSigma == catvalue)
            locations_kc = np.where((z[:, 0] == clusterno) & (combinedcategoriesSigma == catvalue) )
            n_kc = np.sum(count_kc)

            if n_kc > 0:  
                Phivec2= np.mean(continuousdata[locations_kc], axis=0) - mu_cs0_all[catvaluetoindexmu[catvalue], :]

                Phivec1 = continuousdata[locations_kc] - np.mean(continuousdata[locations_kc], axis=0)  
                Phi_new = Phi_new + Phivec1.T @ Phivec1                                                      
                Phi_new=Phi_new+(beta_kg_prior*n_kc/(beta_kg_prior+n_kc))*np.outer(Phivec2,Phivec2.T)

            Sigma[iteration, clusterno, catvaluetoindexSigma[catvalue], :] = invwishart.rvs(df=nu + n_kc, scale=Phi_new, size=1, random_state=rng)  


    for clusterno in range(K):
        for catvalue in allcatvalsmu[0]:
            count_kc = (z[:, 0] == clusterno) & (combinedcategoriesmu == catvalue) 
            locations_kc = np.where((z[:, 0] == clusterno) & (combinedcategoriesmu == catvalue) )
            n_kc = np.sum(count_kc)

            mu_cs0temp = mu_cs0_all[catvaluetoindexmu[catvalue], :]
            Sigma_cs0temp = Sigma[iteration, clusterno, catvaluetoindexmu[g], :]/beta_kg_prior
            if n_kc == 0:  

                mu_cs0new = mu_cs0temp
                Sigma_cs0new = Sigma_cs0temp
            else:

                mu_cs0new=(beta_kg_prior*mu_cs0_all[catvaluetoindexmu[catvalue], :]+np.sum(continuousdata[locations_kc[0],:], axis=0) )/(beta_kg_prior+n_kc)
                Sigma_cs0new = Sigma[iteration, clusterno, catvaluetoindexmu[g], :]/(beta_kg_prior+n_kc)

            mu[iteration, clusterno, catvaluetoindexmu[catvalue], :] = multivariate_normal.rvs(mean=mu_cs0new, cov=Sigma_cs0new, size=1) 


    alphatempPi = np.zeros(K)
    for clusterno in range(K):
        count_k = (z[:, 0] == clusterno) 
        n_k = np.sum(count_k)
        alphatempPi[clusterno] = piparamprior + n_k


    piprob[iteration, :] = rng.dirichlet(alpha=alphatempPi, size=1)

    logpi = np.log(piprob[iteration, :])                       
    zlogprobmatrix = np.empty((N, K))
    for clusterno in range(K):
        zlogprobmatrix[:, clusterno] = multivariate_normal.logpdf(
            continuousdata,
            mean=mu[iteration, clusterno, 0, :],
            cov=Sigma[iteration, clusterno, 0, :])


    for j in range(0, p):
        zlogprobmatrix += np.log(Psi[iteration, :, j, :])[:, catcodes[:, j]].T   


    zlogprobmatrix += logpi[None, :]

    zexpprobmatrix = np.exp(zlogprobmatrix)
    rowsum = zexpprobmatrix.sum(axis=1)
    underflow = rowsum == 0                                     
    if np.any(underflow):
        zlogprobmatrix[underflow] -= zlogprobmatrix[underflow].max(axis=1, keepdims=True)
        zexpprobmatrix[underflow] = np.exp(zlogprobmatrix[underflow])
        rowsum[underflow] = zexpprobmatrix[underflow].sum(axis=1)
    zprobnormalised[:, :] = zexpprobmatrix / rowsum[:, None]

    zuniforms = rng.random(N)
    zcdf = np.cumsum(zprobnormalised, axis=1)
    zcdf /= zcdf[:, -1][:, None]
    z[:, 0] = (zcdf <= zuniforms[:, None]).sum(axis=1)

    gibbs_compute_elapsed += time.perf_counter() - iter_t0     
    timerecord[iteration] = gibbs_compute_elapsed              

    z_history[iteration] = z[:, 0]

    if iteration % 10 == 1:
        ztemp = z.astype(int)
        print('predicted hard clusters')
        print(Counter(ztemp[:,0]))

        print('true clusters')
        print(Counter(fulldataframe['clusterlabel']))

    outputmu = open('mu'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(mu, outputmu)
    outputmu.close()

    outputzprob = open('zprob'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(zprobnormalised, outputzprob)
    outputzprob.close()

    outputz = open('z'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(z, outputz)
    outputz.close()

    outputPsi = open('Psi'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(Psi, outputPsi)
    outputPsi.close()

    outputSigma = open('Sigma'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(Sigma, outputSigma)
    outputSigma.close()

    outputpiprob = open('piprob'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(piprob, outputpiprob)
    outputpiprob.close()

    outputiteration = open('iteration'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(iteration, outputiteration)
    outputiteration.close()

    outputtimerecord = open('timerecord'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(timerecord, outputtimerecord)
    outputtimerecord.close()

    outputjointdistval = open('jointdistval'+str(dataseedtrain)+'N'+str(N)+'q'+str(q)+'Ktrue'+str(Ktrue)+'.pkl', 'wb')
    pickle.dump(jointdistval, outputjointdistval)
    outputjointdistval.close()

    jointdistval[iteration]=computejoint(z=z[:, 0],mu=mu[iteration,:,0,:],Sigma=Sigma[iteration,:,0,:],Psi=Psi[iteration,:,:,:],pi=piprob[iteration,:])
    print('joint density val')
    print(jointdistval[iteration])    


end_time = time.time()
elapsed_time = end_time - start_time

print('elapsed_time')
print(elapsed_time)


In [ ]:


from scipy.optimize import linear_sum_assignment

def error_permuted(truevals, estimatedvals):
    
    
    distance_matrix = np.zeros((Ktrue, K))

    
    for i in range(0,Ktrue):
        for j in range(0,K):
            
            
            distance_matrix[i, j] = np.mean( np.abs(truevals[i,:] - estimatedvals[j,:]) )
            
            
    row_ind, col_ind = linear_sum_assignment(distance_matrix)

    
    matched_true_means = truevals[row_ind,:]
    matched_estimated_means = estimatedvals[col_ind,:]
    
    
    mean_error = np.mean( np.abs(matched_true_means - matched_estimated_means) )

    
    print(f"Matched True Means:\n{matched_true_means}")
    print(f"Matched Estimated Means:\n{matched_estimated_means}")
    
    print(f"Mean Error: {mean_error:.10f}")
    
    print(f"true_ind: {row_ind}")
    print(f"estimated_ind: {col_ind}")

    
    return mean_error, row_ind, col_ind, matched_true_means, matched_estimated_means

def error_nopermute(truevals, estimatedvals, row_ind, col_ind):
    
    
    matched_true_means = truevals[row_ind,:]
    matched_estimated_means = estimatedvals[col_ind,:]
    
    
    mean_error = np.mean( np.abs(matched_true_means - matched_estimated_means) )

    
    print(f"Matched True Means:\n{matched_true_means}")
    print(f"Matched Estimated Means:\n{matched_estimated_means}")
    
    print(f"Mean Error: {mean_error:.10f}")
    
    print(f"true_ind: {row_ind}")
    print(f"estimated_ind: {col_ind}")

    
    return mean_error, row_ind, col_ind, matched_true_means, matched_estimated_means


burnin=burn_in
lastiter=iterations-1

muposteriormean=np.mean(mu[burnin:lastiter,:,0,:], axis=0) 
Sigmaposteriormean=np.mean(Sigma[burnin:lastiter,:,0,:], axis=0)  
 
print(np.shape(muposteriormean))
print(np.shape(Sigmaposteriormean))

print(muposteriormean)

print('mean in the original scale')

error_permuted(truemu_kg[:,0,:], scaler.inverse_transform(muposteriormean) )

print('mean in the standardised scale')

mu_error, mu_row_ind, mu_col_ind, matched_true_mu, matched_estimated_mu=error_permuted(scaler.transform(truemu_kg[:,0,:]), muposteriormean)


print('covariance in the original scale')
estimatedcovogscale=np.array([ np.diag(scaler.scale_) @ Sigmaposteriormean[k,:] @ np.diag(scaler.scale_) for k in range(K)])

error_permuted(trueSigma_k[:,0,:], estimatedcovogscale )

print('covariance in the standardised scale')
trueSigma_kscaled=np.array([ np.diag(1/scaler.scale_) @ trueSigma_k[k, catvaluetoindexmu[g], :] @ np.diag(1/scaler.scale_) for k in range(Ktrue)])
    

sigma_error, sigma_row_ind, sigma_col_ind, matched_true_sigma, matched_estimated_sigma=error_nopermute(trueSigma_kscaled, Sigmaposteriormean, mu_row_ind, mu_col_ind) 

def most_frequent_per_column(arr):
    n_rows, n_cols = arr.shape
    most_frequent = np.zeros(n_cols, dtype=int)  

    for col in range(n_cols):
        col_values = arr[:, col]
        unique_values, counts = np.unique(col_values, return_counts=True)
        most_frequent[col] = unique_values[np.argmax(counts)]  

    return most_frequent

from scipy.stats import mode

mapping = dict(zip(mu_col_ind, mu_row_ind))


predictedclusters=z[:,0] 


orderedclusters=np.vectorize(lambda x: mapping.get(x, -1))(predictedclusters)

print(np.sum(orderedclusters == component)/len(orderedclusters) )

hardclustermatchesprob=np.sum(orderedclusters == component)/len(orderedclusters)

predpsiprob=np.zeros((Ktrue,p,2))


Psiposteriormean=np.mean(Psi[burnin:lastiter,:,:,:], axis=0) 

Psi_error=np.zeros(p)

for j in range(0,p):
    print('we are at column '+str(j))
    
    Psiparamsum=np.sum(Psiposteriormean[mu_col_ind,j,:], axis=1)
    print('truepsi')
    print( truepsi[:,j,:] )
    print('rearranged Psiparam prob')
    print( Psiposteriormean[mu_col_ind,j,:]/Psiparamsum[:,np.newaxis] )
    
    print('error')
    Psi_error[j]=np.mean( np.abs(truepsi[:,j,:]-Psiposteriormean[mu_col_ind,j,:]/Psiparamsum[:,np.newaxis]) )
    
    
    print(Psi_error[j])
    
    
piposteriormean=np.mean(piprob[burnin:lastiter,mu_col_ind], axis=0) 

estimateclusterprob=piposteriormean 


print(piposteriormean)
print(trueclusterprob)
print(np.abs(trueclusterprob-estimateclusterprob))

pi_error=np.mean(np.abs(trueclusterprob-estimateclusterprob)) 

print(pi_error)

mu_error_vec=mu_error
mu_row_ind_vec=mu_row_ind
mu_col_ind_vec=mu_col_ind

sigma_error_vec=sigma_error
sigma_row_ind_vec=sigma_row_ind
sigma_col_ind_vec=sigma_col_ind

#banana


outputmu_error_vec=open('mu_error_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(mu_error_vec, outputmu_error_vec)
outputmu_error_vec.close()

outputmu_row_ind_vec=open('mu_row_ind_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(mu_row_ind_vec, outputmu_row_ind_vec)
outputmu_row_ind_vec.close()

outputmu_col_ind_vec=open('mu_col_ind_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(mu_col_ind_vec, outputmu_col_ind_vec)
outputmu_col_ind_vec.close()

outputsigma_error_vec=open('sigma_error_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(sigma_error_vec, outputsigma_error_vec)
outputsigma_error_vec.close()

outputsigma_row_ind_vec=open('sigma_row_ind_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(sigma_row_ind_vec, outputsigma_row_ind_vec)
outputsigma_row_ind_vec.close()

outputsigma_col_ind_vec=open('sigma_col_ind_vec'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(sigma_col_ind_vec, outputsigma_col_ind_vec)
outputsigma_col_ind_vec.close()

outputhardclustermatchesprob=open('hardclustermatchesprob'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(hardclustermatchesprob, outputhardclustermatchesprob)
outputhardclustermatchesprob.close()

In [ ]:


d=d_1

logposteriortrue=0.0

trueclusterprobfull=np.zeros(K)
trueclusterprobfull[mu_col_ind]=trueclusterprob

truemu_kgfull=np.zeros((K,q))
truemu_kgfull[mu_col_ind,:]=scaler.transform(truemu_kg[:,0,:])

trueSigma_full=np.zeros((K,q,q))
trueSigma_full[mu_col_ind,:]=np.array([ np.diag(1/scaler.scale_) @ trueSigma_k[k, 0, :] @ np.diag(1/scaler.scale_) for k in range(Ktrue)]) 

truepsi_full=np.zeros((K,p,d))
for j in range(0,p):
    truepsi_full[mu_col_ind,j,:]=truepsi[:,j,:]
    


In [ ]:
logposteriortrueall=computejoint(z=component,mu=truemu_kgfull,Sigma=trueSigma_full,Psi=truepsi_full,pi=trueclusterprobfull)
logposteriorall=np.percentile(jointdistval[burn_in:lastiter], 5)

print(logposteriortrueall)
print(logposteriorall)

coverageresults=logposteriortrueall>logposteriorall

print(coverageresults)


outputlogposteriorall=open('logposteriorall'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(logposteriorall, outputlogposteriorall)
outputlogposteriorall.close()

outputlogposteriortrueall=open('logposteriortrueall'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(logposteriortrueall, outputlogposteriortrueall)
outputlogposteriortrueall.close()

outputcoverageresults=open('coverageresults'+str(dataseedtrain)+'.pkl', 'wb')
pickle.dump(coverageresults, outputcoverageresults)
outputcoverageresults.close()

In [ ]:


with open('VIoutput/'+'truemu_kg'+str(dataseedtrain)+'.pkl', 'rb') as file:
    truemu_kg_vi = pickle.load(file)
    
with open('VIoutput/'+'trueSigma_k'+str(dataseedtrain)+'.pkl', 'rb') as file:
    trueSigma_k_vi = pickle.load(file)
    
with open('VIoutput/'+'truepsi'+str(dataseedtrain)+'.pkl', 'rb') as file:
    truepsi_vi = pickle.load(file)
    
with open('VIoutput/'+'trueclusterprob'+str(dataseedtrain)+'.pkl', 'rb') as file:
    trueclusterprob_vi = pickle.load(file)
    
with open('VIoutput/'+'scaler'+str(dataseedtrain)+'.pkl', 'rb') as file:
    scaler_vi = pickle.load(file)

with open('VIoutput/'+'piparam'+str(dataseedtrain)+'.pkl', 'rb') as file:
    piparam_vi = pickle.load(file)


if piparam_vi.ndim == 4: 
    piparam_vi = np.squeeze(piparam_vi, axis=1) 
    
with open('VIoutput/'+'beta_kg_hat'+str(dataseedtrain)+'.pkl', 'rb') as file:
    beta_kg_hat_vi = pickle.load(file)


if beta_kg_hat_vi.ndim == 4: 
    beta_kg_hat_vi = np.squeeze(beta_kg_hat_vi, axis=3) 
    
with open('VIoutput/'+'m_kg_hat'+str(dataseedtrain)+'.pkl', 'rb') as file:
    m_kg_hat_vi = pickle.load(file)


if m_kg_hat_vi.ndim == 5: 
    m_kg_hat_vi = np.squeeze(m_kg_hat_vi, axis=3) 
    
with open('VIoutput/'+'nu_k_hat'+str(dataseedtrain)+'.pkl', 'rb') as file:
    nu_k_hat_vi = pickle.load(file)
    
with open('VIoutput/'+'Phi_k_hat'+str(dataseedtrain)+'.pkl', 'rb') as file:
    Phi_k_hat_vi = pickle.load(file)
    
with open('VIoutput/'+'Psiparam'+str(dataseedtrain)+'.pkl', 'rb') as file:
    Psiparam_vi = pickle.load(file)


if Psiparam_vi.ndim == 5: 
    Psiparam_vi = np.squeeze(Psiparam_vi, axis=3) 
    
with open('VIoutput/'+'r_ik'+str(dataseedtrain)+'.pkl', 'rb') as file:
    r_ik_vi = pickle.load(file)
    
with open('VIoutput/'+'iteration'+str(dataseedtrain)+'.pkl', 'rb') as file:
    iteration_vi = pickle.load(file)
    
with open('VIoutput/'+'mu_col_ind_vec'+str(dataseedtrain)+'.pkl', 'rb') as file:
    mu_col_ind_vec_vi = pickle.load(file)
    
print('done')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde, norm, beta, multivariate_t, invwishart

def plot_density_curves(
    samples_list=None,
    trueval_list=None,
    dist_type='gaussian',  
    dist_params=None,      
    bw_method=None,
    alpha=0.3,
    show_hist=False,
    hist_bins=30,
    title=None,
    filename=None,
    xlabel="Value",
    likelihood_params=None
):
    """
    Plot empirical and matching theoretical densities, using subplots if needed.
    - samples_list: list of 1D arrays (empirical samples)
    - dist_type: string, one of 'gaussian', 'beta', 'multivariate_t', 'inverse_wishart'
    - dist_params: list of dicts, one per empirical distribution, with model parameters
    """
    samples_list = samples_list or []
    dist_params = dist_params or []
    n = len(samples_list)
    colors = plt.cm.tab10.colors

    
    if n > 1:
        fig, axes = plt.subplots(n, 1, figsize=(8, 3*n)) 
        if n == 1:
            axes = [axes]
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        axes = [ax]

    for i, (samples, params, truevals) in enumerate(zip(samples_list, dist_params, trueval_list)):
        ax = axes[i]
        samples = np.asarray(samples)
        samples = samples[~np.isnan(samples)]
        
        color = colors[0 % len(colors)]

        
        lower_bound = np.percentile(samples, 0.0) 
        upper_bound = np.percentile(samples, 100.0) 
        
        
        x_min=lower_bound
        x_max=upper_bound

        x = np.linspace(x_min, x_max, 1000)

        
        if len(samples):
            kde = gaussian_kde(samples, bw_method=bw_method)
            if x is not None:
                density = kde(x)
                ax.plot(x, density, color=color, lw=2, label='Gibbs')
                ax.fill_between(x, density, alpha=alpha, color=color)
            if show_hist:
                ax.hist(samples, bins=hist_bins, density=True, alpha=alpha*0.5, color=color)

        
        if dist_type == 'gaussian':
            mu = params['mean']
            var = params['variance']
            pdf = norm.pdf(x, mu, np.sqrt(var))
            ax.plot(x, pdf, color='orange', linestyle='--', lw=2, label='variational posterior') 
        elif dist_type == 'beta':
            a = params['a']
            b = params['b']
            pdf = beta.pdf(x, a, b)
            ax.plot(x, pdf, color='orange', linestyle='--', lw=2, label='variational posterior') 
        elif dist_type == 'multivariate_t':
            df = params['df']
            loc = params['loc']
            shape = params['shape']
            
            
            pdf = multivariate_t.pdf(x, loc=loc, shape=shape, df=df)
            
            ax.plot(x, pdf, color='orange', linestyle='--', lw=2, label='variational posterior') 
        elif dist_type == 'inverse_wishart':
            df = params['df']
            scale = params['scale']
            
            
            pdf = invwishart.pdf(x, df=df, scale=scale)
            
                
            ax.plot(x, pdf, color='orange', linestyle='--', lw=2, label='variational posterior') 
        else:
            raise ValueError(f"Unknown dist_type: {dist_type}")
        
        if np.size(truevals)>1:
            ax.plot(truevals, np.zeros(np.size(truevals)), marker='.', color='red', markersize=0.5, linestyle="None", label='data')
        else:
            ax.plot(truevals, np.zeros(np.size(truevals)), marker='x', color='green', markersize=10, linestyle="None", label='true value')
        
        
        if likelihood_params!=None:
            truedensity = norm.pdf(x, likelihood_params[i]['mean'], np.sqrt(likelihood_params[i]['variance']))
            ax.plot(x, truedensity, color='green', linestyle='--', lw=2, label='true likelihood') 
        
        ax.set_title(title if n == 1 else f"{title}{i+1}")
        ax.set_xlabel(xlabel)
        
        title_dict={'beta', 'multivariate_t', 'inverse_wishart'}
        
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(filename+'.pdf')
    plt.show()

In [ ]:
tdistSigma_vi=Phi_k_hat_vi[np.min((12,iteration_vi))-1,:,0,0,0]/(nu_k_hat_vi[iteration_vi,:,0]-q+1)
tdistSigma_vi=tdistSigma_vi/beta_kg_hat_vi[iteration_vi,:,0] 
print(tdistSigma_vi)


mu_dicts_params = [{'df': nu_k_hat_vi[iteration_vi,k,0]-q+1, 'loc': m_kg_hat_vi[np.min((12,iteration_vi))-1,k,0,0], 'shape': tdistSigma_vi[k]} for k in range(0,Ktrue)] 

plot_density_curves(samples_list=[mu[burn_in:lastiter,0,0,0],mu[burn_in:lastiter,1,0,0],mu[burn_in:lastiter,2,0,0]], trueval_list=truemu_kgfull[:,0] 
                   ,dist_type='multivariate_t', dist_params=mu_dicts_params, title=r"posterior of $\mu$, k=", filename="mu")

In [ ]:
Sigma_dicts_params = [{'df': nu_k_hat_vi[iteration_vi,k,0], 'scale': Phi_k_hat_vi[np.min((12,iteration_vi))-1,k,0,0,0] } for k in range(0,Ktrue)]

plot_density_curves(samples_list=[Sigma[burn_in:lastiter,0,0,0,0],Sigma[burn_in:lastiter,1,0,0,0],Sigma[burn_in:lastiter,2,0,0,0]], trueval_list=trueSigma_full[:,0,0]
                   ,dist_type='inverse_wishart', dist_params=Sigma_dicts_params, title=r"posterior of $\Sigma$, k=", filename="Sigma")

In [ ]:
pi_dicts_params = [{'a': piparam_vi[iteration_vi,k,0] , 'b': np.sum(piparam_vi[iteration_vi,:,0])-piparam_vi[iteration_vi,k,0] } for k in range(0,Ktrue)] 

plot_density_curves(samples_list=[piprob[burn_in:lastiter,0],piprob[burn_in:lastiter,1],piprob[burn_in:lastiter,2] ], trueval_list=trueclusterprobfull 
                   ,dist_type='beta', dist_params=pi_dicts_params, title=r"posterior of $\pi$, k=", filename="pi")

In [ ]:
for k in range(0,K):

    Psi_dicts_params = [{'a': Psiparam_vi[iteration_vi,k,0,g] , 'b': np.sum(Psiparam_vi[iteration_vi,k,0,:])-Psiparam_vi[iteration_vi,k,0,g] } for g in range(0,d_1)] 

    plot_density_curves(samples_list=[Psi[burn_in:lastiter,k,0,0],Psi[burn_in:lastiter,k,0,1],Psi[burn_in:lastiter,k,0,2] ], trueval_list=truepsi_full[k,0,:] 
                       ,dist_type='beta', dist_params=Psi_dicts_params, title=r"posterior of $\Psi$, k="+str(k+1)+" g=", filename="Psi_kis"+str(k+1))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, multivariate_t

plt.rcParams['text.usetex'] = True

def plot_mixture_density_combined(
    weights_gauss,
    params_gauss,
    weights_t,
    params_t,
    empirical_data=None,
    num_points=1000,
    addtitle=""
):
    """
    Plot the density of a mixture of multivariate Gaussians and t-distributions on the same graph,
    with optional empirical data points (in red).

    Parameters
    ----------
    weights_gauss : list or array
        Mixture weights for Gaussian components (should sum to 1).
    params_gauss : list of dicts
        Each dict: {'loc': mean, 'shape': covariance}
    weights_t : list or array
        Mixture weights for t-distribution components (should sum to 1).
    params_t : list of dicts
        Each dict: {'df': degrees_of_freedom, 'loc': mean, 'shape': covariance}
    empirical_data : array-like, optional
        Data points to plot as red ticks.
    num_points : int
        Number of points on the x-axis.
    """
    weights_gauss = np.array(weights_gauss)
    weights_t = np.array(weights_t)
    assert np.isclose(weights_gauss.sum(), 1), "Gaussian weights must sum to 1"
    assert np.isclose(weights_t.sum(), 1), "t-distribution weights must sum to 1"

    
    lowerrange=[p['loc']-2*np.sqrt(p['shape']) for p in params_gauss + params_t]
    higherrange=[p['loc']+2*np.sqrt(p['shape']) for p in params_gauss + params_t]
    
    centresmin = min(lowerrange)
    centresmax = max(higherrange)
    
    print('check centresmin')
    print(centresmin)
    print('check centresmax')
    print(centresmax)

    if empirical_data is not None and len(empirical_data) > 0:
        lower_bound = np.percentile(empirical_data, 0.0)
        upper_bound = np.percentile(empirical_data, 100.0)
        xmin = min(lower_bound, centresmin)
        xmax = max(upper_bound, centresmax)
    else:
        xmin = centresmin
        xmax = centresmax

    x = np.linspace(xmin, xmax, num_points)
    grid = x[:, np.newaxis]

    pdf_gauss = np.zeros(num_points, dtype=float)
    pdf_t = np.zeros(num_points, dtype=float)

    
    for w, p in zip(weights_gauss, params_gauss):
        loc = np.atleast_1d(p['loc'])
        shape = np.atleast_2d(p['shape'])
        rv = multivariate_normal(mean=loc, cov=shape)
        pdf_gauss += w * rv.pdf(grid)

    
    for w, p in zip(weights_t, params_t):
        df = p['df']
        loc = np.atleast_1d(p['loc'])
        shape = np.atleast_2d(p['shape'])
        rv = multivariate_t(loc=loc, shape=shape, df=df)
        pdf_t += w * rv.pdf(grid)

    plt.figure(figsize=(8, 4))
    plt.plot(x, pdf_gauss, label=r"True $p(x|c,\mu^{*},\Lambda^{*},\psi^{*},\pi^{*})$", color='green')
    plt.plot(x, pdf_t, label=r"Variational posterior predictive $q(\tilde{x}|\tilde{c},x,c)$", color='orange', linestyle='--')
    plt.xlabel('x')
    plt.ylabel('Density')
    plt.title("Distribution of x given c="+str(addtitle))
    plt.legend()

    if empirical_data is not None:
        plt.scatter(empirical_data, np.zeros_like(empirical_data), color='red', marker='.', s=1, label='Empirical Data')
        plt.legend()
    
    plt.savefig("xgivenc"+str(addtitle)+".pdf")
    plt.show()


In [ ]:
#use if you get a latex error

plt.rcParams['text.usetex'] = False


In [ ]:
likelihood_params=[{'loc': scaler.transform(truemu_kg[:,0,:])[k,0], 'shape':  1/scaler.scale_[0] * trueSigma_k[k, 0, 0, 0] * 1/scaler.scale_[0]  } for k in range(0,Ktrue)]

cval=0 #value of c, change to 0 or 1 or 2 for the value you want
trueweights=[trueclusterprobfull[k]*truepsi_full[k,0,cval] for k in range(0,Ktrue)]
trueweights=trueweights/np.sum(trueweights)

piweights_vi=[(piparam_vi[iteration_vi,k,0]/np.sum(piparam_vi[iteration_vi,:,0]) )*(Psiparam_vi[iteration_vi,k,0,cval]/np.sum(Psiparam_vi[iteration_vi,k,0,:]) ) for k in range(0,Ktrue)] 
piweights_vi=piweights_vi/np.sum(piweights_vi)

shapemattop_vi=Phi_k_hat_vi[np.min((12,iteration_vi))-1,:,0,0,0]*(beta_kg_hat_vi[iteration_vi,:,0]+1) 
shapematbottom_vi=beta_kg_hat_vi[iteration_vi,:,0]*(nu_k_hat_vi[iteration_vi,:,0]-q+1) 
postpredshape_vi=shapemattop_vi/shapematbottom_vi

vi_postpredx_params = [{'df': nu_k_hat_vi[iteration_vi,k,0]-q+1, 'loc': m_kg_hat_vi[np.min((12,iteration_vi))-1,k,0,0], 'shape': postpredshape_vi[k]} for k in range(0,Ktrue)] 

plot_mixture_density_combined(weights_gauss=piweights_vi, params_gauss=likelihood_params, weights_t=piweights_vi, params_t=vi_postpredx_params, empirical_data=fulldataframe[fulldataframe['catcol_1']==cval]['contcol_1'],addtitle=str(cval))

In [ ]:
pctrue=np.zeros(3)
qcvi=np.zeros(3)
empiricalc=np.zeros(3)

for cval2 in range(0,3):
    pctrue[cval2]=np.sum([trueclusterprobfull[k]*truepsi_full[k,0,cval2] for k in range(0,Ktrue)])
    qcvi[cval2]=np.sum([(piparam_vi[iteration_vi,k,0]/np.sum(piparam_vi[iteration_vi,:,0]) )*(Psiparam_vi[iteration_vi,k,0,cval2]/np.sum(Psiparam_vi[iteration_vi,k,0,:]) ) for k in range(0,Ktrue)])
    empiricalc[cval2]=fulldataframe[fulldataframe['catcol_1']==cval2]['contcol_1'].shape[0]/N
    
print(pctrue)
print(qcvi)
print(empiricalc)

In [ ]:
bar_width = 0.18
x = np.arange(d_1)  


plt.bar(x - 1.0*bar_width, qcvi, width=bar_width, label=r"variational posterior predictive $q(\tilde{c}|x,c)$", alpha=0.8)
plt.bar(x + 0.0*bar_width, pctrue, width=bar_width, label=r"true $p(c)$", alpha=0.8)
plt.bar(x + 1.0*bar_width, empiricalc, width=bar_width, label='empirical distribution of c', alpha=0.8)
plt.title(f"posterior predictive of c", fontsize=12)
plt.xticks(x, labels=[f'Category {i+1}' for i in range(d_1)])
plt.ylabel('Probability')
plt.ylim(0, 0.5)
plt.grid(axis='y', alpha=0.3)

plt.legend(loc='upper right')

plt.savefig('posteriorpredc'+'.pdf')